<center>
<a href="https://www.umontpellier.fr/"><img src="https://www.umontpellier.fr/wp-content/uploads/2022/10/logo_um_2022_rouge_rvb.svg" width="200"/></a>&nbsp;&nbsp;
<a href="https://economie.edu.umontpellier.fr/"><img src="https://economie.edu.umontpellier.fr/files/2014/12/economie_rvb_2015-300x137.png" width="160"/></a>
</center>

<div align="center">

#  Phase 5 — Analyse Économétrique

| Nom et Prénom | Rôle |
|---|---|
| Randriamisaina Tsiory-Fanomezana | Membre de l'équipe |
| SHIRALI POUR Amir | Membre de l'équipe |

</div>

---
## Objectif

Identifier et quantifier les **déterminants de la durée de traitement** d'un dossier d'assistance.

**Variable dépendante :** `duree_totale_h` (durée totale en heures)  
**Approche :** Régression OLS (Ordinary Least Squares) avec validation des hypothèses classiques  
**Référence :** [`docs/phase5_econometrie.md`](../docs/phase5_econometrie.md)


---
## Section 0 — Initialisation

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.stats as stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

def localiser_racine_du_projet():
    """Remonte l'arborescence jusqu'à trouver la racine du projet."""
    repertoire_courant = Path.cwd()
    while True:
        if any((repertoire_courant / m).exists() for m in ['.git', 'requirements.txt']):
            break
        if repertoire_courant.parent == repertoire_courant:
            break
        repertoire_courant = repertoire_courant.parent
    if Path.cwd() != repertoire_courant:
        os.chdir(repertoire_courant)
    return repertoire_courant.resolve()

REPERTOIRE_RACINE = localiser_racine_du_projet()
sys.path.insert(0, str(REPERTOIRE_RACINE / 'src'))
from utils.dataframe_styler import style_duplicates
print(f" Racine du projet : {REPERTOIRE_RACINE}")

In [ ]:
#  Chargement du dataset complet 
dataset = pd.read_csv('data/dataset_complet.csv', encoding='utf-8')

print(f" Dataset : {dataset.shape[0]:,} lignes × {dataset.shape[1]} colonnes")
print(f"Lignes avec duree_totale_h non-NaN : {dataset['duree_totale_h'].notna().sum():,}")
print()
print("Variables numériques disponibles :")
print(dataset.select_dtypes(include='number').columns.tolist())

---
## Section 1 — Statistiques Descriptives

### 1.1 Variable dépendante : `duree_totale_h`

In [ ]:
print("Statistiques de duree_totale_h :")
print(dataset['duree_totale_h'].describe().round(3))
print()
print(f"Asymétrie (skewness) : {dataset['duree_totale_h'].skew():.3f}")
print(f"Aplatissement (kurtosis) : {dataset['duree_totale_h'].kurt():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Distribution de la variable dépendante : duree_totale_h", fontsize=13, fontweight='bold')

duree_valide = dataset['duree_totale_h'].dropna()
duree_p99    = duree_valide[duree_valide <= duree_valide.quantile(0.99)]

# Histogramme original
axes[0].hist(duree_p99, bins=60, color='#4472c4', edgecolor='white')
axes[0].set_title("Distribution originale (P99)")
axes[0].set_xlabel("Durée (heures)")
axes[0].set_ylabel("Fréquence")

# Transformation log
log_duree = np.log1p(duree_valide)
axes[1].hist(log_duree, bins=60, color='#ed7d31', edgecolor='white')
axes[1].set_title("Distribution log(1 + duree_totale_h)")
axes[1].set_xlabel("log(1 + durée)")
axes[1].set_ylabel("Fréquence")

# Q-Q plot de log_duree
stats.probplot(log_duree, dist='norm', plot=axes[2])
axes[2].set_title("Q-Q plot (log-transformée)")

plt.tight_layout()
plt.savefig('data/phase5_distribution_variable_dependante.png', dpi=150, bbox_inches='tight')
plt.show()

skewness_original = duree_valide.skew()
skewness_log      = log_duree.skew()
print(f"Asymétrie originale     : {skewness_original:.3f}")
print(f"Asymétrie log-transformée: {skewness_log:.3f}")
print()
if abs(skewness_log) < abs(skewness_original):
    print("→ La transformation log réduit l'asymétrie — on l'utilisera dans les modèles OLS")
    UTILISER_LOG = True
else:
    print("→ La transformation log n'améliore pas — on utilise la variable originale")
    UTILISER_LOG = True  # Forcé car skewness originale > 1 dans ce dataset

### 1.2 Variables indépendantes numériques

In [ ]:
colonnes_numeriques = [
    'agent_experience_j', 'agent_duree_travail_j', 'agent_temps_travail_pct',
    'delai_survenance_ouverture_j', 'nb_interventions', 'nb_agents_distincts'
]
print("Statistiques des variables indépendantes numériques :")
dataset[colonnes_numeriques].describe().round(2)

### 1.3 Variables catégorielles

In [ ]:
colonnes_categorielles = ['agent_lieu_travail', 'agent_contrat', 'agent_population',
                            'Cause.intervention', 'Type.d.energie', 'annee_ouverture']
for col in colonnes_categorielles:
    print(f"{col} :")
    print(dataset[col].value_counts(dropna=False).head(6).to_string())
    print()

---
## Section 2 — Matrice de Corrélation

In [ ]:
colonnes_correlation = [
    'duree_totale_h', 'agent_experience_j', 'agent_duree_travail_j',
    'delai_survenance_ouverture_j', 'nb_interventions', 'nb_agents_distincts',
    'mois_ouverture'
]

df_corr = dataset[colonnes_correlation].dropna()
matrice_correlation = df_corr.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(matrice_correlation, dtype=bool))
sns.heatmap(
    matrice_correlation,
    mask=mask,
    annot=True, fmt='.3f',
    cmap='RdYlGn', center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax
)
ax.set_title("Matrice de corrélation des variables numériques", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('data/phase5_matrice_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Corrélations avec duree_totale_h (triées) :")
corr_avec_cible = matrice_correlation['duree_totale_h'].drop('duree_totale_h').abs().sort_values(ascending=False)
print(corr_avec_cible.round(4))

---
## Section 3 — Préparation des Données pour l'OLS

### Décisions appliquées :
- **E1 :** `dropna()` sur toutes les colonnes utilisées
- **E2 :** Transformation `log(1 + duree_totale_h)` (asymétrie > 1)
- **E3 :** Modalités de référence : CDI, SITE, CAS
- **E4 :** Seuil α = 5%

In [ ]:
#  Sélection des colonnes pour la modélisation 
colonnes_modelisation = [
    'duree_totale_h',
    'agent_experience_j', 'agent_duree_travail_j',
    'delai_survenance_ouverture_j', 'nb_interventions', 'nb_agents_distincts',
    'agent_lieu_travail', 'agent_contrat', 'agent_population',
    'mois_ouverture', 'annee_ouverture'
]

df_ols = dataset[colonnes_modelisation].dropna().copy()

#  Transformation logarithmique 
df_ols['log_duree_totale_h'] = np.log1p(df_ols['duree_totale_h'])

#  Dummies avec modalités de référence explicites 
# La modalité de référence est la première (drop_first=True) → on réordonne
df_ols['agent_lieu_travail'] = pd.Categorical(df_ols['agent_lieu_travail'],
                                               categories=['SITE', 'TELE'], ordered=False)
df_ols['agent_contrat']      = pd.Categorical(df_ols['agent_contrat'],
                                               categories=['CDI', 'CDD', 'CDS'], ordered=False)
df_ols['agent_population']   = pd.Categorical(df_ols['agent_population'],
                                               categories=['CAS', 'CAC'], ordered=False)
df_ols['annee_ouverture']    = df_ols['annee_ouverture'].astype(str)

print(f" Dataset pour modélisation : {len(df_ols):,} observations")
print(f"   (vs {len(dataset):,} total — {len(dataset)-len(df_ols):,} exclus pour NaN)")
print()
print("Distribution de log_duree_totale_h :")
print(df_ols['log_duree_totale_h'].describe().round(3))

---
## Section 4 — Régression OLS Simple (modèle de référence)

**Modèle :** `log(1 + duree_totale_h) ~ agent_experience_j`

Ce modèle simple permet de valider que l'expérience de l'agent a un effet significatif
sur la durée de traitement avant d'introduire d'autres variables.

In [ ]:
#  Modèle OLS simple 
formule_simple = 'log_duree_totale_h ~ agent_experience_j'
modele_simple  = smf.ols(formule_simple, data=df_ols).fit()

print(modele_simple.summary())

In [ ]:
#  Visualisation de la relation 
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("OLS Simple : log(duree_totale_h) ~ agent_experience_j", fontsize=12, fontweight='bold')

# Nuage de points + droite de régression
echantillon = df_ols.sample(min(5000, len(df_ols)), random_state=42)
axes[0].scatter(echantillon['agent_experience_j'], echantillon['log_duree_totale_h'],
                alpha=0.2, s=5, color='#4472c4')
x_range = np.linspace(df_ols['agent_experience_j'].min(), df_ols['agent_experience_j'].max(), 100)
y_pred  = modele_simple.params['Intercept'] + modele_simple.params['agent_experience_j'] * x_range
axes[0].plot(x_range, y_pred, color='red', linewidth=2)
axes[0].set_xlabel("Expérience (jours)")
axes[0].set_ylabel("log(1 + duree_totale_h)")
axes[0].set_title(f"R² = {modele_simple.rsquared:.4f}")

# Résidus
axes[1].scatter(modele_simple.fittedvalues, modele_simple.resid,
                alpha=0.2, s=5, color='#4472c4')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel("Valeurs ajustées")
axes[1].set_ylabel("Résidus")
axes[1].set_title("Graphique résidus vs ajustés")

plt.tight_layout()
plt.savefig('data/phase5_ols_simple.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"R² = {modele_simple.rsquared:.4f} | R² ajusté = {modele_simple.rsquared_adj:.4f}")
print(f"Coeff. agent_experience_j : {modele_simple.params['agent_experience_j']:.6f} (p={modele_simple.pvalues['agent_experience_j']:.4f})")

---
## Section 5 — Régression OLS Multiple

**Modèle complet** avec toutes les variables candidates.

In [ ]:
#  Formule OLS multiple 
formule_multiple = (
    'log_duree_totale_h ~ '
    'agent_experience_j + agent_duree_travail_j + delai_survenance_ouverture_j '
    '+ nb_interventions + nb_agents_distincts '
    '+ C(agent_lieu_travail) + C(agent_contrat) + C(agent_population) '
    '+ mois_ouverture + C(annee_ouverture)'
)

modele_multiple = smf.ols(formule_multiple, data=df_ols).fit()
print(modele_multiple.summary())

In [ ]:
print(f"R² = {modele_multiple.rsquared:.4f} | R² ajusté = {modele_multiple.rsquared_adj:.4f}")
print(f"F-statistic : {modele_multiple.fvalue:.2f} (p = {modele_multiple.f_pvalue:.6f})")
print()
print("Variables significatives (p < 0.05) :")
pvalues_significatives = modele_multiple.pvalues[modele_multiple.pvalues < 0.05].sort_values()
print(pvalues_significatives.round(4))

---
## Section 6 — Tests des Hypothèses Classiques

### 6a — Normalité des Résidus (Shapiro-Wilk + Q-Q plot)

**Hypothèse H₀ :** les résidus suivent une distribution normale.  
Si p-value > 0.05 → on ne rejette pas H₀ → hypothèse validée.

In [ ]:
residus = modele_multiple.resid

# Test de Shapiro-Wilk (sur échantillon de 5000 si trop grand)
echantillon_residus = residus.sample(min(5000, len(residus)), random_state=42)
stat_shapiro, pvalue_shapiro = stats.shapiro(echantillon_residus)
print(f"Test de Shapiro-Wilk (n={len(echantillon_residus):,}) :")
print(f"  Statistique W = {stat_shapiro:.6f}")
print(f"  p-value = {pvalue_shapiro:.6f}")
print(f"  → {'H₀ non rejetée — résidus normaux ' if pvalue_shapiro > 0.05 else 'H₀ rejetée — résidus non normaux '}")
print()

# Test de Jarque-Bera (robuste sur grands échantillons)
res_jb = stats.jarque_bera(residus); stat_jb, pvalue_jb = res_jb.statistic, res_jb.pvalue; skew_jb = float(residus.skew()); kurt_jb = float(residus.kurt())
print(f"Test de Jarque-Bera (n={len(residus):,}) :")
print(f"  Statistique JB = {stat_jb:.2f}")
print(f"  p-value = {pvalue_jb:.6f}")
print(f"  Asymétrie = {skew_jb:.4f} | Aplatissement = {kurt_jb:.4f}")
print(f"  → {'H₀ non rejetée — résidus normaux ' if pvalue_jb > 0.05 else 'H₀ rejetée  (attendu sur grands échantillons)'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("6a — Test de Normalité des Résidus", fontsize=12, fontweight='bold')

# Histogramme des résidus
axes[0].hist(residus, bins=80, color='#4472c4', edgecolor='white', density=True)
x_norm = np.linspace(residus.min(), residus.max(), 200)
axes[0].plot(x_norm, stats.norm.pdf(x_norm, residus.mean(), residus.std()),
             'r-', linewidth=2, label='Normale théorique')
axes[0].set_title("Distribution des résidus")
axes[0].set_xlabel("Résidus")
axes[0].legend()

# Q-Q plot
stats.probplot(residus, dist='norm', plot=axes[1])
axes[1].set_title("Q-Q plot des résidus")

plt.tight_layout()
plt.savefig('data/phase5_normalite_residus.png', dpi=150, bbox_inches='tight')
plt.show()

### 6b — Homoscédasticité (Test de Breusch-Pagan)

**Hypothèse H₀ :** la variance des résidus est constante (homoscédasticité).  
Si p-value > 0.05 → on ne rejette pas H₀ → pas d'hétéroscédasticité.

In [ ]:
# Test de Breusch-Pagan
exog_bp = sm.add_constant(modele_multiple.model.exog)
stat_bp, pvalue_bp, _, _ = het_breuschpagan(modele_multiple.resid, exog_bp)

print(f"Test de Breusch-Pagan :")
print(f"  Statistique LM = {stat_bp:.4f}")
print(f"  p-value = {pvalue_bp:.6f}")
print(f"  → {'H₀ non rejetée — homoscédasticité ' if pvalue_bp > 0.05 else 'H₀ rejetée — hétéroscédasticité détectée '}")
print()

# Graphique résidus² vs valeurs ajustées
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(modele_multiple.fittedvalues, modele_multiple.resid**2,
           alpha=0.2, s=4, color='#4472c4')
ax.axhline(y=(modele_multiple.resid**2).mean(), color='red', linestyle='--', label='Variance moyenne')
ax.set_title("6b — Résidus² vs Valeurs ajustées (Breusch-Pagan)")
ax.set_xlabel("Valeurs ajustées")
ax.set_ylabel("Résidus²")
ax.legend()
plt.tight_layout()
plt.savefig('data/phase5_homoscedasticite.png', dpi=150, bbox_inches='tight')
plt.show()

### 6c — Absence de Multicolinéarité (VIF)

**Règle :** VIF > 5 → multicolinéarité modérée ; VIF > 10 → multicolinéarité sévère.

In [ ]:
# Calcul du VIF pour chaque variable numérique
colonnes_vif = [
    'agent_experience_j', 'agent_duree_travail_j',
    'delai_survenance_ouverture_j', 'nb_interventions',
    'nb_agents_distincts', 'mois_ouverture'
]

df_vif = df_ols[colonnes_vif].dropna()
X_vif  = sm.add_constant(df_vif)

resultats_vif = pd.DataFrame({
    'Variable': colonnes_vif,
    'VIF': [variance_inflation_factor(X_vif.values, i+1) for i in range(len(colonnes_vif))]
}).sort_values('VIF', ascending=False)

resultats_vif['Interprétation'] = resultats_vif['VIF'].apply(
    lambda v: ' Acceptable (< 5)' if v < 5 else (' Modéré (5-10)' if v < 10 else ' Sévère (> 10)')
)

print("Facteurs d'Inflation de la Variance (VIF) :")
print(resultats_vif.to_string(index=False))

### 6d — Absence d'Autocorrélation (Durbin-Watson)

**Règle :** valeur ≈ 2 → pas d'autocorrélation ; < 1.5 ou > 2.5 → autocorrélation détectée.

In [ ]:
stat_dw = durbin_watson(modele_multiple.resid)

print(f"Test de Durbin-Watson :")
print(f"  Statistique DW = {stat_dw:.4f}")
if 1.5 <= stat_dw <= 2.5:
    print(f"  → Pas d'autocorrélation significative ")
elif stat_dw < 1.5:
    print(f"  → Autocorrélation positive détectée ")
else:
    print(f"  → Autocorrélation négative détectée ")

---
## Section 7 — Synthèse et Interprétation Économétrique

In [ ]:
#  Tableau de synthèse des tests 
bilan_tests = pd.DataFrame([
    {'Test': 'Shapiro-Wilk',   'H₀': 'Résidus normaux',      'p-value': f'{pvalue_shapiro:.4f}',
     'Résultat': 'Non rejetée ' if pvalue_shapiro > 0.05 else 'Rejetée '},
    {'Test': 'Jarque-Bera',    'H₀': 'Résidus normaux',      'p-value': f'{pvalue_jb:.4f}',
     'Résultat': 'Non rejetée ' if pvalue_jb > 0.05 else 'Rejetée (normal sur grands n)'},
    {'Test': 'Breusch-Pagan',  'H₀': 'Homoscédasticité',     'p-value': f'{pvalue_bp:.4f}',
     'Résultat': 'Non rejetée ' if pvalue_bp > 0.05 else 'Rejetée '},
    {"Test": "Durbin-Watson", "H0": "Pas d'autocorrelation", "p-value": f"DW={stat_dw:.3f}",
     'Résultat': 'Non rejetée ' if 1.5 <= stat_dw <= 2.5 else 'Rejetée '},
])
print("Bilan des tests des hypothèses classiques :")
bilan_tests

In [ ]:
#  Résumé final du modèle retenu 
print("=== RÉSUMÉ DU MODÈLE OLS MULTIPLE ===")
print()
print(f"Variable dépendante  : log(1 + duree_totale_h)")
print(f"Observations         : {int(modele_multiple.nobs):,}")
print(f"R²                   : {modele_multiple.rsquared:.4f}")
print(f"R² ajusté            : {modele_multiple.rsquared_adj:.4f}")
print(f"F-statistique        : {modele_multiple.fvalue:.2f} (p = {modele_multiple.f_pvalue:.2e})")
print()
print("Coefficients significatifs (p < 0.05) :")
coefs_sig = pd.DataFrame({
    'Coefficient': modele_multiple.params,
    'Std. Error' : modele_multiple.bse,
    'p-value'    : modele_multiple.pvalues
})
coefs_sig = coefs_sig[coefs_sig['p-value'] < 0.05].sort_values('p-value')
print(coefs_sig.round(6).to_string())
print()
print("→ Interprétation : un coefficient positif indique que cette variable")
print("  augmente la durée de traitement (en log-heures).")

In [ ]:
#  Sauvegarde du résumé OLS 
with open('data/phase5_ols_summary.txt', 'w') as f:
    f.write(str(modele_multiple.summary()))

print(" Résumé OLS sauvegardé : data/phase5_ols_summary.txt")